# Fine-tune armenia-lawyer-router and export to GGUF

Retrains the Armenian legal-consultant model on the conversational answer format in `src/data/finetune_dataset.csv` (see `notebook/build_finetune_dataset.py`), then exports straight to GGUF via Unsloth's own exporter -- which applies fixes for Gemma-2-specific GGUF quirks that generic `llama.cpp` conversion has been observed to get wrong for this checkpoint.

**Before running:**
1. Kaggle notebook settings -> Accelerator -> **GPU T4 x2** (or any CUDA GPU). Unsloth requires CUDA; it will not run on a CPU-only or TPU instance.
2. Settings -> Internet -> **on** (needed to pip install and to pull the base model from Hugging Face).
3. Add Data -> upload `src/data/finetune_dataset.csv` from this repo as a Kaggle Dataset, then attach it to this notebook. Update `DATASET_CSV_PATH` below if your dataset slug differs.
4. **Do not skip the sanity-check cell before export.** A previous run of this same config (700 steps, fp16 on a T4, no checkpointing, lr=2e-4) silently diverged and exported weights that produced pure noise -- scattered letters, no real words -- at every quantization level tested. That failure is why this version halves the learning rate and checkpoints/evals periodically: if it diverges again, you can load an earlier `outputs/checkpoint-<step>` instead of exporting ruined weights.

In [ ]:
%%capture
# Single, unconflicting install -- letting unsloth's own dependency spec pick
# compatible trl/transformers/peft/accelerate versions together. An earlier
# version of this cell installed unsloth+trl unpinned, then force-downgraded
# trl separately with --no-deps, which caused a PicklingError during
# checkpoint saving ("Can't pickle trl.trainer.sft_config.SFTConfig: it's not
# the same object as trl.trainer.sft_config.SFTConfig").
#
# That same error can ALSO recur even with a clean install, because Unsloth
# writes a dynamically-generated trainer file to
# /kaggle/working/unsloth_compiled_cache/ on disk -- and that directory is
# NOT cleared by "Restart & Clear Cell Outputs" (that only restarts the
# Python process; /kaggle/working persists across restarts within the same
# session). If a previous attempt compiled that cache against a different
# trl/transformers version than what's currently installed, every run in
# this session reuses the stale, mismatched file. Deleting it here forces a
# fresh compile against whatever this cell just installed.
!rm -rf /kaggle/working/unsloth_compiled_cache
!pip install -q --upgrade pip
!pip install -q -U unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # dataset answers now top out around 900 chars (legal-basis + ruling excerpt, not full documents), so 1024 covers them with headroom -- see build_finetune_dataset.py
dtype = None  # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-2b-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

In [ ]:
import glob
import os

# Edit this if your uploaded Kaggle dataset has a different slug/filename.
DATASET_CSV_PATH = "/kaggle/input/finetune-dataset/finetune_dataset.csv"
if not os.path.exists(DATASET_CSV_PATH):
    matches = glob.glob("/kaggle/input/**/finetune_dataset.csv", recursive=True)
    if not matches:
        raise FileNotFoundError(
            "finetune_dataset.csv not found under /kaggle/input. "
            "Upload src/data/finetune_dataset.csv from the repo as a Kaggle Dataset and attach it to this notebook."
        )
    DATASET_CSV_PATH = matches[0]
print("Using dataset:", DATASET_CSV_PATH)

In [ ]:
from datasets import load_dataset

# Same instruction framing used in the pre-export sanity check below, so the
# model learns the exact prompt shape it will be evaluated (and served) with.
INSTRUCTION_PREFIX = (
    "### Instruction:\nԴուք հայ իրավաբան եք (legal consultant)։ Պատասխանեք հաճախորդի հարցին հստակ, մարդամոտ և իրավական հիմքով։\n\nQuery: "
)

raw = load_dataset("csv", data_files=DATASET_CSV_PATH)["train"]

def format_example(ex):
    ex["text"] = f"{INSTRUCTION_PREFIX}{ex['query']}\n\n### Response:\n{ex['answer']}" + tokenizer.eos_token
    return ex

raw = raw.map(format_example)
dataset = raw.train_test_split(test_size=0.05, seed=3407)
print(dataset)
print(dataset["train"][0]["text"][:500])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# TrainingArguments, hardened after a run on this same config (700 steps,
# fp16 on a T4, no checkpointing) silently diverged: exported weights that
# produced pure noise (scattered letters, no real words) at EVERY
# quantization level tested (Q4_K_M, Q8_0) -- confirming it was the training
# itself that broke, not the GGUF export/quantization step. Two things made
# that failure both more likely and unrecoverable:
#   - learning_rate=2e-4 sustained over 700 steps (vs. the original 120) on
#     much longer target sequences (roughly 450-900 chars vs. ~100 for the
#     old routing triples) is a harder, longer run at the same aggressive LR
#     that worked fine for the short/easy original task. Halved below.
#   - save_strategy="no" meant if the run diverged at, say, step 400, there
#     was no way to recover the good step-400 weights -- only the final,
#     ruined ones got saved and exported. Now checkpoints periodically and
#     evaluates periodically so divergence is visible (and recoverable)
#     during training instead of discovered only after a failed export.
training_args = TrainingArguments(
    per_device_train_batch_size=2,      # safer on T4
    gradient_accumulation_steps=4,      # effective batch size = 8
    warmup_steps=5,
    max_steps=700,
    learning_rate=1e-4,  # halved from 2e-4 -- lower risk of divergence over this much longer/harder run
    max_grad_norm=1.0,   # explicit gradient clipping -- HF's default, but made visible since it matters here
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",

    # Key fixes for the 'int' .mean() error
    average_tokens_across_devices=False,
    dataloader_drop_last=False,

    # Checkpoint + eval periodically -- if training diverges late, you can
    # still load the last good checkpoint from outputs/ instead of losing
    # the whole run. save_total_limit keeps disk usage bounded.
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=50,
)

# Explicit data collator (helps stability with Gemma-2)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding="longest",
    return_tensors="pt"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"] if "test" in dataset else None,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
    data_collator=data_collator,
)

In [ ]:
trainer_stats = trainer.train()
print("Training completed successfully!")
# Watch the eval_loss column printed during training (every eval_steps): it
# should trend down and stay finite. A sudden jump to a huge number or "nan"
# is training divergence -- if you see that, use the checkpoint from BEFORE
# it happened (outputs/checkpoint-<step>) instead of the final model below.

In [ ]:
# --- Sanity check BEFORE exporting to GGUF ---
# Catches a diverged/broken model immediately, before spending time on GGUF
# conversion and a multi-GB download, by generating one real sample and
# reading it. If this prints scattered letters / non-words instead of real
# Armenian sentences, do NOT proceed to the export cell below -- the run
# diverged. Load an earlier checkpoint from outputs/checkpoint-<step>
# instead (FastLanguageModel.from_pretrained(model_name="outputs/checkpoint-XXX", ...))
# and re-run this same check on it before exporting.
FastLanguageModel.for_inference(model)
test_prompt = dataset["train"][0]["query"] if "query" in dataset["train"].column_names else "\u053b\u0576\u0579\u057a\u0565\u057d \u056f\u0561\u0580\u0578\u0572 \u0565\u0574 \u057e\u056b\u0573\u0561\u0580\u056f\u0565\u056c \u0561\u0577\u056d\u0561\u057f\u0561\u0576\u0584\u056b\u0581 \u0561\u0566\u0561\u057f\u0574\u0561\u0576 \u0578\u0580\u0578\u0577\u0578\u0582\u0574\u0568:"
inputs = tokenizer(
    f"{INSTRUCTION_PREFIX}{test_prompt}\n\n### Response:\n",
    return_tensors="pt",
).to(model.device)
output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# --- Export to GGUF ---
# Only run this once the sanity-check cell above printed a real, coherent
# Armenian response. If it printed scattered letters/non-words instead, this
# would just export the same broken weights in three different quantization
# levels -- go back, load an earlier checkpoint from outputs/checkpoint-<step>,
# re-run the sanity check on it, and only export once that looks right.
model.save_pretrained_gguf(
    "armenia_lawyer_router_gguf",
    tokenizer,
    quantization_method=["q4_k_m", "q5_k_m", "q8_0"]
)

# Or just one common one:
# model.save_pretrained_gguf("armenia_lawyer_router", tokenizer, quantization_method="q4_k_m")

In [ ]:
import os

folder = "armenia_lawyer_router_gguf"

print("\u2705 GGUF files created in folder:", folder)
print("-" * 60)

if os.path.exists(folder):
    files = [f for f in os.listdir(folder) if f.endswith(".gguf")]
    if files:
        for f in sorted(files):
            size_gb = os.path.getsize(os.path.join(folder, f)) / (1024 ** 3)
            print(f"\u2022 {f:50} \u2192 {size_gb:.2f} GB")
    else:
        print("No .gguf files found!")
else:
    print(f"Folder '{folder}' not found!")

## Using the new GGUF locally

1. Download the `.gguf` you want (the deployed app uses **q4_k_m**, ~1.7GB, to fit an 8GB Mac's memory budget -- see Kaggle's notebook output/Data tab) to your machine.
2. Replace the broken file: `armenia-lawyer-router.Q4_K_M.gguf` in this repo's root, or place the new one alongside `Modelfile` and update its `FROM` path if the filename differs.
3. Rebuild the Ollama model: `ollama create armenia-lawyer-router -f Modelfile` (from the repo root -- this overwrites the existing, broken model).
4. Sanity-check it directly before trusting the app again:
   ```bash
   curl http://localhost:11434/api/generate -d '{"model":"armenia-lawyer-router","prompt":"What is 2+2?","stream":false,"options":{"num_predict":30}}'
   ```
   A real sentence back means it's good; scattered single letters means this run diverged too -- go back to an earlier checkpoint on Kaggle and retry from the sanity-check cell.
5. Restart the backend (`uvicorn api:app --reload --port 8000`) and test a real message in the chat UI.